# 第16章：混合精度策略 -- FP16/FP32/TF32/BF16

## 前置知识
- 第09章：分块矩阵乘法基础
- 基本的浮点数表示知识
- 了解 GEMM 中的累加器概念

## 学习目标
- 理解不同浮点精度的**表示范围**和**精度**
- 理解为什么 GEMM 需要 **FP32 累加器**
- 掌握 Triton 中 `tl.dot` 的精度控制参数
- 理解 BF16 vs FP16 的权衡
- 实测不同精度配置对**准确度**和**性能**的影响

## 对应 CUDA 概念
- CUDA 中的 `__half` (FP16), `__nv_bfloat16` (BF16)
- WMMA/MMA 指令中的精度选择
- cuBLAS 的计算类型 (CUBLAS_COMPUTE_16F, CUBLAS_COMPUTE_32F, etc.)

In [1]:
import torch
import triton
import triton.language as tl
import struct

## 16.1 浮点精度层级

### 常见浮点格式

```
格式      位数   指数位  尾数位  表示范围         精度 (十进制)
────────────────────────────────────────────────────────────────
FP32      32     8       23     +-3.4e38         ~7 位
TF32      19     8       10     +-3.4e38         ~3 位
BF16      16     8       7      +-3.4e38         ~2 位
FP16      16     5       10     +-6.5e4          ~3 位
FP8(E4M3) 8      4       3      +-448            ~1 位
FP8(E5M2) 8      5       2      +-5.7e4          ~0.5 位
```

### 位布局对比

```
FP32:  [S|EEEEEEEE|MMMMMMMMMMMMMMMMMMMMMMM]  (1+8+23 = 32 bits)
         1   8              23

TF32:  [S|EEEEEEEE|MMMMMMMMMM]               (1+8+10 = 19 bits)
         1   8         10
         ↑ 指数与 FP32 相同! 只截断尾数

BF16:  [S|EEEEEEEE|MMMMMMM]                   (1+8+7 = 16 bits)
         1   8        7
         ↑ 指数与 FP32 相同! 范围大,精度低

FP16:  [S|EEEEE|MMMMMMMMMM]                   (1+5+10 = 16 bits)
         1  5       10
         ↑ 指数位少 → 范围小 (容易溢出!)
           尾数位多 → 精度比 BF16 高
```

### BF16 vs FP16 的关键区别

```
FP16:               BF16:
  范围: +-65504       范围: +-3.4e38
  精度: ~3位          精度: ~2位
  
  优势: 精度更高       优势: 范围更大
  劣势: 容易溢出       劣势: 精度略低
  
  适合: 需要精度的场景  适合: 需要大范围的场景 (如 训练中的梯度)
  
  BF16 本质上是 FP32 的直接截断 → FP32→BF16→FP32 转换几乎无损
  FP16 需要重新映射指数 → FP32→FP16 可能溢出!
```

In [2]:
# ========== 演示不同精度的范围和精度 ==========
print("浮点精度对比:")
print(f"{'格式':>10} | {'最大值':>15} | {'最小正值':>15} | {'1.0+eps':>15}")
print("-" * 65)

for dtype, name in [
    (torch.float32, "FP32"),
    (torch.bfloat16, "BF16"),
    (torch.float16, "FP16"),
]:
    info = torch.finfo(dtype)
    print(f"{name:>10} | {info.max:>15.4e} | {info.tiny:>15.4e} | {info.eps:>15.4e}")

print("\n精度演示:")
x = torch.tensor(1000.0)
for dtype, name in [(torch.float32, "FP32"), (torch.bfloat16, "BF16"), (torch.float16, "FP16")]:
    x_cast = x.to(dtype)
    x_back = x_cast.float()
    err = abs(x_back.item() - x.item())
    print(f"  {name}: 1000.0 → {x_cast.item()} (误差: {err})")

print("\n溢出演示:")
x = torch.tensor(70000.0)
for dtype, name in [(torch.float32, "FP32"), (torch.bfloat16, "BF16"), (torch.float16, "FP16")]:
    x_cast = x.to(dtype)
    print(f"  {name}: 70000.0 → {x_cast.item()}")

浮点精度对比:
        格式 |             最大值 |            最小正值 |         1.0+eps
-----------------------------------------------------------------
      FP32 |      3.4028e+38 |      1.1755e-38 |      1.1921e-07
      BF16 |      3.3895e+38 |      1.1755e-38 |      7.8125e-03
      FP16 |      6.5504e+04 |      6.1035e-05 |      9.7656e-04

精度演示:
  FP32: 1000.0 → 1000.0 (误差: 0.0)
  BF16: 1000.0 → 1000.0 (误差: 0.0)
  FP16: 1000.0 → 1000.0 (误差: 0.0)

溢出演示:
  FP32: 70000.0 → 70000.0
  BF16: 70000.0 → 70144.0
  FP16: 70000.0 → inf


## 16.2 为什么 GEMM 需要 FP32 累加器？

### 累加误差分析

在 GEMM 的 K 循环中，我们做了 K 次乘加：

```
C[i,j] = Σ_{k=0}^{K-1} A[i,k] * B[k,j]

如果使用 FP16 累加器:
  每次乘法: FP16 * FP16 → FP16 (精度 ~3位)
  每次加法: FP16 + FP16 → FP16 (可能溢出!)
  
  K=1024 次累加后:
  - 单次误差 ~eps * |result| ≈ 1e-3 * |result|
  - 累积误差 ~sqrt(K) * eps * |result| ≈ 32 * 1e-3 = 0.032
  - 这是 相对误差 3.2%! 非常大!

如果使用 FP32 累加器:
  每次乘法: FP16 * FP16 → FP32 (Tensor Core 原生支持)
  每次加法: FP32 + FP32 → FP32 (精度 ~7位)
  
  K=1024 次累加后:
  - 累积误差 ~sqrt(K) * eps_fp32 ≈ 32 * 1.2e-7 ≈ 4e-6
  - 相对误差 0.0004%  — 可以接受!
```

### 数值实验

```
典型 GEMM 误差 (K=1024, 随机矩阵):

输入精度   累加器    相对误差 (L2)     说明
────────────────────────────────────────────────
FP16      FP16      ~1e-2 (1%)       不可接受!
FP16      FP32      ~1e-4 (0.01%)    标准做法
BF16      FP32      ~1e-3 (0.1%)     可接受
FP32      FP32      ~1e-7            完美
TF32      FP32      ~1e-4            良好
```

In [3]:
# ========== 累加误差实验 ==========
torch.manual_seed(42)
K = 1024

# 随机 FP16 向量
a = torch.randn(K, device='cuda', dtype=torch.float16)
b = torch.randn(K, device='cuda', dtype=torch.float16)

# FP32 参考值
ref = (a.float() * b.float()).sum().item()

# FP16 累加
acc_fp16 = torch.tensor(0.0, device='cuda', dtype=torch.float16)
for i in range(K):
    acc_fp16 += a[i] * b[i]
err_fp16 = abs(acc_fp16.float().item() - ref) / abs(ref)

# FP32 累加
acc_fp32 = torch.tensor(0.0, device='cuda', dtype=torch.float32)
for i in range(K):
    acc_fp32 += a[i].float() * b[i].float()
err_fp32 = abs(acc_fp32.item() - ref) / abs(ref)

# 向量化 FP16 累加 (torch sum)
err_vec = abs((a * b).sum().float().item() - ref) / abs(ref)

print(f"K={K} 次乘加的累加误差:")
print(f"  FP32 累加:     相对误差 = {err_fp32:.2e}")
print(f"  FP16 累加:     相对误差 = {err_fp16:.2e}  (差 {err_fp16/max(err_fp32,1e-15):.0f} 倍)")
print(f"  FP16 向量化:   相对误差 = {err_vec:.2e}")
print(f"  参考值: {ref:.4f}")

K=1024 次乘加的累加误差:
  FP32 累加:     相对误差 = 3.27e-07
  FP16 累加:     相对误差 = 1.80e-03  (差 5490 倍)
  FP16 向量化:   相对误差 = 2.14e-04
  参考值: 23.3075


## 16.3 Triton 中的精度控制

### tl.dot 的精度参数

```python
# 基本形式
result = tl.dot(a, b, acc=acc)

# 精度控制 (Triton 2.x+)
result = tl.dot(a, b, acc=acc, allow_tf32=True)
#                                allow_tf32: 允许使用 TF32 Tensor Core
#                                  True  → 更快, 精度 ~3位
#                                  False → 使用 FP16/BF16 Tensor Core
```

### 输入精度 + 累加器精度的组合

```
输入 dtype    allow_tf32    硬件操作              累加器
──────────────────────────────────────────────────────────
float16      False         FP16 Tensor Core      FP32
float16      True          TF32 Tensor Core      FP32 (FP16先转FP32再截TF32)
bfloat16     False         BF16 Tensor Core      FP32
bfloat16     True          TF32 Tensor Core      FP32
float32      False         FP32 CUDA Core (慢)    FP32
float32      True          TF32 Tensor Core      FP32
```

### Tensor Core 支持的精度

```
GPU 架构       支持的输入精度                Tensor Core 版本
──────────────────────────────────────────────────────────────
Volta (V100)   FP16                         v1
Turing (T4)    FP16, INT8, INT4             v2
Ampere (A100)  FP16, BF16, TF32, INT8      v3
Hopper (H100)  FP16, BF16, TF32, FP8, INT8 v4
```

## 16.4 实现：多精度 GEMM Kernel

In [4]:
@triton.jit
def matmul_precision_kernel(
    a_ptr, b_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
    GROUP_SIZE_M: tl.constexpr,
    # 精度控制
    ALLOW_TF32: tl.constexpr,
    # 输出精度 (由 host 端类型决定)
    OUTPUT_FP32: tl.constexpr,
):
    """
    支持多种精度配置的 GEMM kernel。
    输入: a, b 的精度由调用者决定 (FP16 或 BF16)
    累加: 始终使用 FP32
    输出: FP16/BF16 或 FP32 (由 OUTPUT_FP32 控制)
    """
    # Swizzled pid
    pid = tl.program_id(0)
    grid_m = tl.cdiv(M, BLOCK_M)
    grid_n = tl.cdiv(N, BLOCK_N)
    group_id = pid // (GROUP_SIZE_M * grid_n)
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(grid_m - first_pid_m, GROUP_SIZE_M)
    pid_m = first_pid_m + ((pid % (GROUP_SIZE_M * grid_n)) % group_size_m)
    pid_n = (pid % (GROUP_SIZE_M * grid_n)) // group_size_m
    
    a_block_ptr = tl.make_block_ptr(
        base=a_ptr, shape=(M, K), strides=(stride_am, stride_ak),
        offsets=(pid_m * BLOCK_M, 0), block_shape=(BLOCK_M, BLOCK_K), order=(1, 0),
    )
    b_block_ptr = tl.make_block_ptr(
        base=b_ptr, shape=(K, N), strides=(stride_bk, stride_bn),
        offsets=(0, pid_n * BLOCK_N), block_shape=(BLOCK_K, BLOCK_N), order=(1, 0),
    )
    
    # 累加器始终为 FP32
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    
    for k in range(0, K, BLOCK_K):
        a_tile = tl.load(a_block_ptr, boundary_check=(0, 1))
        b_tile = tl.load(b_block_ptr, boundary_check=(0, 1))
        
        # tl.dot: allow_tf32 控制是否使用 TF32 Tensor Core
        acc = tl.dot(a_tile, b_tile, acc=acc, allow_tf32=ALLOW_TF32)
        
        a_block_ptr = tl.advance(a_block_ptr, (0, BLOCK_K))
        b_block_ptr = tl.advance(b_block_ptr, (BLOCK_K, 0))
    
    # 输出精度选择
    if OUTPUT_FP32:
        c = acc  # 保持 FP32
    else:
        c = acc.to(tl.float16)  # 转为 FP16
    
    c_block_ptr = tl.make_block_ptr(
        base=c_ptr, shape=(M, N), strides=(stride_cm, stride_cn),
        offsets=(pid_m * BLOCK_M, pid_n * BLOCK_N),
        block_shape=(BLOCK_M, BLOCK_N), order=(1, 0),
    )
    tl.store(c_block_ptr, c, boundary_check=(0, 1))

In [5]:
def matmul_precision(a, b, allow_tf32=False, output_fp32=False,
                     BLOCK_M=128, BLOCK_N=128, BLOCK_K=32, GROUP_SIZE_M=8):
    """多精度 GEMM 包装函数。"""
    M, K = a.shape
    K2, N = b.shape
    assert K == K2
    
    out_dtype = torch.float32 if output_fp32 else a.dtype
    c = torch.empty((M, N), device=a.device, dtype=out_dtype)
    grid = (triton.cdiv(M, BLOCK_M) * triton.cdiv(N, BLOCK_N),)
    
    matmul_precision_kernel[grid](
        a, b, c, M, N, K,
        a.stride(0), a.stride(1), b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K,
        GROUP_SIZE_M=GROUP_SIZE_M,
        ALLOW_TF32=allow_tf32,
        OUTPUT_FP32=output_fp32,
    )
    return c

In [6]:
# ========== 精度对比实验 ==========
torch.manual_seed(42)
M, N, K = 2048, 2048, 1024

# FP64 参考值
a_fp32 = torch.randn(M, K, device='cuda', dtype=torch.float32)
b_fp32 = torch.randn(K, N, device='cuda', dtype=torch.float32)
c_ref_fp64 = torch.matmul(a_fp32.double(), b_fp32.double())  # 高精度参考

a_fp16 = a_fp32.half()
b_fp16 = b_fp32.half()
a_bf16 = a_fp32.bfloat16()
b_bf16 = b_fp32.bfloat16()

print("不同精度配置的误差分析 (相对于 FP64 参考值):")
print(f"矩阵规模: M={M}, N={N}, K={K}")
print(f"\n{'配置':>30} | {'相对误差(L2)':>15} | {'最大绝对误差':>15}")
print("-" * 70)

configs = [
    ("FP16 + FP32累加", a_fp16, b_fp16, False),
    ("FP16 + TF32", a_fp16, b_fp16, True),
    ("BF16 + FP32累加", a_bf16, b_bf16, False),
    ("BF16 + TF32", a_bf16, b_bf16, True),
]

for name, a, b, tf32 in configs:
    try:
        c = matmul_precision(a, b, allow_tf32=tf32, output_fp32=True)
        rel_err = torch.norm(c.double() - c_ref_fp64) / torch.norm(c_ref_fp64)
        max_err = (c.double() - c_ref_fp64).abs().max().item()
        print(f"{name:>30} | {rel_err.item():>15.6e} | {max_err:>15.4f}")
    except Exception as e:
        print(f"{name:>30} | ERROR: {str(e)[:40]}")

# cuBLAS 参考
c_cublas_fp16 = torch.matmul(a_fp16, b_fp16)
rel_err_cublas = torch.norm(c_cublas_fp16.double() - c_ref_fp64) / torch.norm(c_ref_fp64)
max_err_cublas = (c_cublas_fp16.double() - c_ref_fp64).abs().max().item()
print(f"{'cuBLAS FP16':>30} | {rel_err_cublas.item():>15.6e} | {max_err_cublas:>15.4f}")

不同精度配置的误差分析 (相对于 FP64 参考值):
矩阵规模: M=2048, N=2048, K=1024

                            配置 |        相对误差(L2) |          最大绝对误差
----------------------------------------------------------------------


                 FP16 + FP32累加 |    2.933738e-04 |          0.0536
                   FP16 + TF32 |    2.933738e-04 |          0.0536
                 BF16 + FP32累加 |    2.348657e-03 |          0.4340
                   BF16 + TF32 |    2.348657e-03 |          0.4340
                   cuBLAS FP16 |    3.594841e-04 |          0.0866


## 16.5 BF16 vs FP16 深入对比

### 何时选择 BF16？

```
BF16 的指数范围与 FP32 相同 (8 位指数):
  → 不会出现 FP16 的溢出问题 (FP16 max = 65504)
  → FP32 → BF16 的转换是安全的 (只截断尾数)
  → 非常适合训练 (梯度可能很大)

FP16 的精度更高 (10 位尾数 vs 7 位):
  → 对精度敏感的推理任务更好
  → 但范围小, 需要 loss scaling

实践指南:
  训练:      BF16 (安全, 不需要 loss scaling)
  推理:      FP16 (精度更高)
  大模型:    BF16 (参数值可能很大)
  量化后:    FP16 (值范围已被控制)
```

In [7]:
# ========== BF16 vs FP16 的溢出实验 ==========
print("BF16 vs FP16 溢出对比:")
print()

# 使用较大值的矩阵
torch.manual_seed(42)
M, N, K = 512, 512, 1024

for scale in [1.0, 10.0, 100.0, 250.0]:
    a_fp32 = torch.randn(M, K, device='cuda') * scale
    b_fp32 = torch.randn(K, N, device='cuda') * scale
    c_ref = torch.matmul(a_fp32.double(), b_fp32.double())
    
    # FP16
    a_fp16 = a_fp32.half()
    b_fp16 = b_fp32.half()
    has_inf_input = a_fp16.isinf().any() or b_fp16.isinf().any()
    if not has_inf_input:
        c_fp16 = torch.matmul(a_fp16, b_fp16)
        has_inf_output = c_fp16.isinf().any().item()
        fp16_err = torch.norm(c_fp16.double() - c_ref).item() / torch.norm(c_ref).item() if not has_inf_output else float('inf')
    else:
        fp16_err = float('inf')
        has_inf_output = True
    
    # BF16
    a_bf16 = a_fp32.bfloat16()
    b_bf16 = b_fp32.bfloat16()
    c_bf16 = torch.matmul(a_bf16, b_bf16)
    bf16_err = torch.norm(c_bf16.double() - c_ref).item() / torch.norm(c_ref).item()
    
    print(f"  scale={scale:>5.0f}: "
          f"FP16 err={'INF/NaN!' if has_inf_output else f'{fp16_err:.4e}':>12}, "
          f"BF16 err={bf16_err:.4e}")

BF16 vs FP16 溢出对比:



  scale=    1: FP16 err=  3.5914e-04, BF16 err=2.8725e-03
  scale=   10: FP16 err=  3.5991e-04, BF16 err=2.8742e-03
  scale=  100: FP16 err=    INF/NaN!, BF16 err=2.8711e-03
  scale=  250: FP16 err=    INF/NaN!, BF16 err=2.8696e-03


In [8]:
# ========== 性能对比: 不同精度 ==========
def benchmark_precision(a, b, allow_tf32=False, num_warmup=10, num_rep=50):
    for _ in range(num_warmup):
        matmul_precision(a, b, allow_tf32=allow_tf32)
    torch.cuda.synchronize()
    
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(num_rep):
        matmul_precision(a, b, allow_tf32=allow_tf32)
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end) / num_rep

M, N, K = 2048, 2048, 2048
flops = 2.0 * M * N * K

print(f"性能对比 (M={M}, N={N}, K={K}):")
print(f"{'配置':>25} | {'时间(ms)':>10} | {'TFLOPS':>8}")
print("-" * 50)

for name, dtype, tf32 in [
    ("FP16", torch.float16, False),
    ("FP16 + TF32", torch.float16, True),
    ("BF16", torch.bfloat16, False),
    ("BF16 + TF32", torch.bfloat16, True),
]:
    try:
        a = torch.randn(M, K, device='cuda', dtype=dtype)
        b = torch.randn(K, N, device='cuda', dtype=dtype)
        ms = benchmark_precision(a, b, allow_tf32=tf32)
        tflops = flops / (ms * 1e-3) / 1e12
        print(f"{name:>25} | {ms:>10.3f} | {tflops:>8.2f}")
    except Exception as e:
        print(f"{name:>25} | ERROR: {str(e)[:30]}")

性能对比 (M=2048, N=2048, K=2048):
                       配置 |     时间(ms) |   TFLOPS
--------------------------------------------------
                     FP16 |      0.142 |   120.95
              FP16 + TF32 |      0.108 |   159.67
                     BF16 | ERROR: at 65:4:
    # 输出精度选择
    if O
              BF16 + TF32 | ERROR: at 65:4:
    # 输出精度选择
    if O


## 16.x 统一 Shape Set 数值表

为了让 Part 3 的所有优化章节可以横向比较，本表使用与 Ch.12、Ch.13、Ch.14 和 Ch.18 相同的 7 个共享矩阵形状。

本章比较 `Ch.16 fp32 acc`、`Ch.16 tf32 allowed` 与 `cuBLAS`，主指标是 `latency_ms`，并附带 `TFLOPS`、相对 `cuBLAS` 的速度比，以及相对章内基线 `Ch.16 fp32 acc` 的速度比。若某个方法运行失败，会保留为 `NaN/False` 以便直接看到失败情况。

## 16.6 实践指南

### 精度选择决策树

```
你的应用场景是什么?
│
├─ 训练 (需要反向传播)
│  ├─ 标准训练: BF16 输入 + FP32 累加
│  ├─ 大模型训练: BF16 (不需要 loss scaling!)
│  └─ 老 GPU (无BF16): FP16 + FP32 累加 + loss scaling
│
├─ 推理 (只有前向传播)
│  ├─ 精度敏感: FP16 + FP32 累加 (如: 科学计算)
│  ├─ 性能优先: FP16 + TF32 (如: 实时推理)
│  └─ 极致性能: FP8 (如: LLM serving, Hopper GPU)
│
└─ 不确定
   └─ 默认: FP16 + FP32 累加 (最安全的选择)
```

### Triton 中的精度最佳实践

```python
# 推荐: FP16 输入, FP32 累加, FP16 输出
acc = tl.zeros((BM, BN), dtype=tl.float32)  # FP32 累加器
for k in range(...):
    a = tl.load(...)  # FP16
    b = tl.load(...)  # FP16
    acc = tl.dot(a, b, acc=acc)  # FP16 x FP16 → FP32 累加
c = acc.to(tl.float16)  # 最后再转回 FP16
```

In [9]:
import sys
from pathlib import Path

for candidate in (Path.cwd(), Path.cwd() / "03_matmul_optimization"):
    candidate_str = str(candidate)
    if candidate_str not in sys.path:
        sys.path.append(candidate_str)

from benchmark_utils import (
    BENCHMARK_SHAPES,
    add_relative_columns,
    benchmark_method,
    format_results,
    make_fp16_inputs,
)

print("Ch.16 shared-shape benchmark: fp32 acc vs tf32 allowed vs cuBLAS")
chapter16_results = []
previous_map = {"Ch.16 tf32 allowed": "Ch.16 fp32 acc"}

for shape in BENCHMARK_SHAPES:
    a, b = make_fp16_inputs(shape.M, shape.N, shape.K)
    c_ref = torch.matmul(a, b)
    methods = {
        "Ch.16 fp32 acc": lambda x, y: matmul_precision(
            x, y, allow_tf32=False, output_fp32=False
        ),
        "Ch.16 tf32 allowed": lambda x, y: matmul_precision(
            x, y, allow_tf32=True, output_fp32=False
        ),
        "cuBLAS": lambda x, y: torch.matmul(x, y),
    }
    for method_name, fn in methods.items():
        chapter16_results.append(
            benchmark_method(method_name, fn, shape, a, b, c_ref=c_ref)
        )
    del a, b, c_ref
    torch.cuda.empty_cache()

chapter16_df = add_relative_columns(
    format_results(chapter16_results),
    previous_method_by_method=previous_map,
)
chapter16_df[[
    "shape_name",
    "category",
    "method",
    "latency_ms",
    "tflops",
    "speedup_vs_cublas",
    "speedup_vs_previous",
    "max_err",
    "passed",
]]

Ch.16 shared-shape benchmark: fp32 acc vs tf32 allowed vs cuBLAS


,shape_name,category,method,latency_ms,tflops,speedup_vs_cublas,speedup_vs_previous,max_err,passed
0,square-2k,square,Ch.16 fp32 acc,0.065406,262.663427,0.961741,NaN,0.00,True
1,square-2k,square,Ch.16 tf32 allowed,0.177392,96.846919,0.354604,0.368711,0.00,True
2,square-2k,square,cuBLAS,0.062904,273.112506,1.000000,NaN,0.00,True
3,square-4k,square,Ch.16 fp32 acc,0.777742,176.715267,1.149789,NaN,0.00,True
4,square-4k,square,Ch.16 tf32 allowed,0.586741,234.241352,1.524080,1.325530,0.00,True
5,square-4k,square,cuBLAS,0.894240,153.693588,1.000000,NaN,0.00,True
6,tall-8k-x-512,tall-skinny,Ch.16 fp32 acc,0.172765,99.440795,0.358842,NaN,0.00,True
7,tall-8k-x-512,tall-skinny,Ch.16 tf32 allowed,0.065214,263.436738,0.950637,2.649182,0.00,True
8,tall-8k-x-512,tall-skinny,cuBLAS,0.061995,277.116108,1.000000,NaN,0.00,True
9,tall-16k-x-256,tall-skinny,Ch.16 fp32 acc,0.065352,262.882077,4.337104,NaN,0.00,True


## 16.7 总结

### 本章要点

1. **浮点精度层级**：FP32 > TF32 > BF16 ~ FP16 > FP8
   - FP16: 精度好 (10位尾数), 范围小 (max=65504)
   - BF16: 精度低 (7位尾数), 范围大 (与FP32相同)
   - TF32: NVIDIA 专有, 截断 FP32 的尾数

2. **FP32 累加器至关重要**：
   - K 维度的大量累加会放大舍入误差
   - FP16 累加误差 ~1%, FP32 累加误差 ~0.01%
   - Tensor Core 原生支持 FP16*FP16→FP32

3. **Triton 精度控制**：
   - `acc = tl.zeros(..., dtype=tl.float32)` — FP32 累加器
   - `allow_tf32=True` — 允许使用 TF32 Tensor Core
   - 输入 dtype 由 host 端的 tensor 类型决定

4. **BF16 vs FP16**：
   - BF16: 训练首选, 不需要 loss scaling, 不怕溢出
   - FP16: 推理首选, 精度更高

### 练习

1. **误差随 K 增长**：固定 M=N=1024, 改变 K (256,512,1024,2048,4096), 画出不同精度配置的误差曲线
2. **FP8 实验** (需要 Hopper GPU)：尝试使用 `tl.float8e4nv` 作为输入精度
3. **Loss Scaling 实现**：实现一个简单的 FP16 + loss scaling 的 GEMM 包装函数
4. **思考题**：为什么 TF32 使用 8 位指数 + 10 位尾数, 而不是其他组合？（提示：兼容 FP32 的指数范围, 同时匹配 Tensor Core 的硬件设计）

### 下一章预告

第17章将深入 **Tensor Core 的工作原理**，探讨 `tl.dot` 如何映射到 `mma.sync` PTX 指令，
以及 Triton 的简洁性相比 CUDA WMMA/MMA API 的巨大优势。